# CleanFrame: Complete Data Cleaning & Preprocessing Guide

**CleanFrame** is an automated and manual data cleaning toolkit for pandas DataFrames designed for data science and ML pipelines.

### Core Features:
- 🔍 **Pre-Cleaning Diagnostic Diagnosis**: Inspect data health, missing cells, outliers, and proposed actions without touching data.
- ❓ **Interactive User Confirmation**: Prompts with proposed actions before executing (`[y/N/customize]`).
- 🔄 **Duplicate Rows**: Drops exact or subset duplicates.
- 📅 **Datetime Feature Extraction**: Auto-detects dates and extracts calendar features (`year`, `month`, `day`, `day_of_week`, `is_weekend`).
- 🩹 **Missing Value Imputation**: Robust median/mean/mode/constant imputation without data leakage.
- 📈 **Outlier Handling**: IQR Tukey fences or Z-scores with winsorization (`clip`), `drop`, or `nan`.
- 🏷️ **Categorical Encoding**: Automatic one-hot vs label encoding with consistent train/test column alignment.
- 💾 **Model Persistence**: `save()` and `load()` fitted pipelines for production inference.

In [ ]:
import pandas as pd
import numpy as np
import cleanframe as cf
from cleanframe import DataFrameCleaner, clean_dataframe, clean_interactive, diagnose_dataframe, load_cleaner

pd.set_option("display.max_columns", 30)

## 1. Create a Realistic Messy Dataset
Contains duplicates, missing values, extreme outliers, dates formatted as strings, and mixed categories.

In [ ]:
df = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 104, 105, 106, 107, 108, 109, 110, 111],
    "signup_date": [
        "2023-01-15 08:30:00", "2023-02-14 12:45:00", "2023-03-22 17:10:00",
        "2023-04-05 09:15:00", "2023-04-05 09:15:00",
        "2023-05-18 20:00:00", "2023-06-30 11:20:00", "2023-07-04 15:50:00",
        "2023-08-19 14:10:00", "2023-09-02 18:30:00", "2023-10-10 10:05:00",
        "2023-11-25 19:40:00"
    ],
    "age": [24.0, 31.0, np.nan, 45.0, 45.0, 29.0, 52.0, 36.0, np.nan, 28.0, 145.0, 33.0],
    "annual_income": [45000, 62000, 58000, 75000, 75000, 51000, 89000, 68000, 72000, 64000, 95000, 1200000],
    "loyalty_tier": ["Silver", "Gold", "Bronze", "Gold", "Gold", None, "Silver", "Platinum", "Bronze", None, "Gold", "Silver"],
    "city": ["New York", "Chicago", "Boston", "Austin", "Austin", "Chicago", "New York", "Seattle", "Denver", "Boston", "Austin", "New York"],
    "churned": [0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0]
})

print("Raw shape:", df.shape, "| Duplicates:", df.duplicated().sum(), "| Missing cells:", df.isna().sum().sum())
df

## 2. Pre-Cleaning Diagnosis (Inspect Before Changing Anything)
Inspect data quality issues, health metrics, and proposed cleaning actions before modifying your data.

In [ ]:
# Pre-cleaning diagnosis:
diagnosis = diagnose_dataframe(df, display=True, target_column="churned")
diagnosis.to_dataframe()

## 3. Interactive Confirmation Mode ("Ask me should I clean or not")
CleanFrame can prompt you with the proposed action plan before applying transformations.

In [ ]:
# Simulating interactive confirmation with user approval ('y'):
cleaner_interactive = DataFrameCleaner(target_column="churned")
clean_df = cleaner_interactive.clean(df, interactive=True, prompt_fn=lambda prompt: "y")
print("Cleaned shape:", clean_df.shape)
clean_df.head()

## 4. Automatic 1-Line Cleanup (Default)
End-to-end automated cleaning pipeline with sensible, production-tested defaults.

In [ ]:
cleaner = DataFrameCleaner(target_column="churned", verbose=True)
cleaned_df = cleaner.clean(df)
cleaned_df.head()

In [ ]:
# View HTML summary card in Jupyter notebook
cleaner

## 5. Manual Mode: Custom Step Selection & Parameters
Run only the steps you want and configure custom imputation, outlier thresholds, and encoding strategies.

In [ ]:
manual_cleaner = DataFrameCleaner(
    steps=["duplicates", "missing", "outliers"],  # Exclude datetime and encoding
    missing_numeric_strategy="mean",
    missing_categorical_strategy="mode",
    outliers_method="zscore",
    outliers_action="clip",
    outliers_zscore_threshold=2.5,
    verbose=True
)
manual_df = manual_cleaner.clean(df)
manual_df.head()

## 6. Granular Step Execution with `df.pipe()`
Every step is accessible as an independent method for custom chaining.

In [ ]:
c = DataFrameCleaner(verbose=False)
piped_df = (
    df
    .pipe(c.remove_duplicates)
    .pipe(c.impute_missing, numeric_strategy="median", categorical_fill_value="Unknown")
    .pipe(c.treat_outliers, method="iqr", action="clip", iqr_factor=1.5)
    .pipe(c.extract_datetime_features, features=["year", "month", "day", "is_weekend"])
)
piped_df.head()

## 7. Machine Learning: Train / Test Split Alignment (No Data Leakage)
Statistics (medians, categories, outlier fences) are learned on train data and applied to test data, ensuring identical feature schemas.

In [ ]:
train_df = df.iloc[:8].copy()
test_df = df.iloc[8:].copy()

ml_cleaner = DataFrameCleaner(target_column="churned", verbose=False)
ml_cleaner.fit(train_df)

X_train = ml_cleaner.transform(train_df)
X_test = ml_cleaner.transform(test_df)

assert list(X_train.columns) == list(X_test.columns)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape, "| Schemas aligned: True")

## 8. Model Persistence: Save & Load for Production Deployment
Serialize the entire fitted cleaner pipeline to disk and reload it in any production API or worker.

In [ ]:
# Save fitted pipeline to disk
ml_cleaner.save("production_cleaner.pkl")

# Load saved pipeline in another process or service
loaded_pipeline = load_cleaner("production_cleaner.pkl")
production_output = loaded_pipeline.transform(test_df)
print("Successfully transformed new batch in production:", production_output.shape)